# multiply-back — ex1: implement multiply_back0 / multiply_back1 with unbroadcast

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `multiply-back`. Running the final beacon cell reports progress against the `Backprop: multiply_back` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: multiply_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`multiply-back`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "multiply-back"
DD_SUBTOPIC = "Backprop: multiply_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## multiply_back0 / multiply_back1 — quick refresher

Binary elementwise op `out = x * y` registers TWO back fns — one per argument position — even though the local derivatives are symmetric:

```
d(x*y)/dx = y          =>  multiply_back0(grad_out, out, x, y) = grad_out * y
d(x*y)/dy = x          =>  multiply_back1(grad_out, out, x, y) = grad_out * x
```

Then wrap the result in `unbroadcast(..., parent)` so the returned grad matches the ORIGINAL (pre-broadcast) shape of the parent.

Two practical wrinkles:
- **Scalar floats on either side.** `multiply(t, 3.0)` must work — coerce the float to a tensor (or just let torch broadcast) so the math doesn't trip on type mismatches.
- **Symmetric registration.** Both bodies are tiny mirror images, but both still get added to `BACK_FUNCS` at argnums 0 and 1 — the dispatcher doesn't know multiply is symmetric, it just looks up `(func, argnum)`.

### Exercise 1 — implement multiply_back0 / multiply_back1 with unbroadcast

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the per-arg-position binary back-fn pattern to write multiply_back0 and multiply_back1, wrapping each result in unbroadcast(grad, parent) so the returned grad matches the pre-broadcast input shape.
> Keywords: multiply-back, binary-op, unbroadcast, back0, back1
> ```

**KCs targeted:** `multiply-back`, `arg-position-back-functions`

Implement TWO back fns for `out = x * y`. Both must:
1. Compute the local derivative w.r.t. the right arg (`d(x*y)/dx = y`, `d(x*y)/dy = x`).
2. Multiply by `grad_out` (the chain rule).
3. **Wrap the result in `unbroadcast(grad, parent)`** so it has the same shape as the parent (pre-broadcast).

We've given you `unbroadcast(grad, original)` already implemented for you in the setup cell — it peels leading axes and collapses size-1 expanded axes via `sum(dim=i, keepdim=True)`.

Signatures:

```python
def multiply_back0(grad_out, out, x, y) -> Tensor:   # dL/dx
def multiply_back1(grad_out, out, x, y) -> Tensor:   # dL/dy
```

**Float-input bonus.** Either of `x` or `y` may be a Python float (`multiply(t, 3.0)` is a valid call). Coerce floats to tensors via `torch.tensor(...)` so `unbroadcast` and the broadcasting math don't trip — OR just let torch broadcast and don't call `unbroadcast` when the parent isn't a tensor.

Inputs are plain `torch.Tensor` (or Python float). No autograd. Return tensors that match the corresponding parent's shape.

In [ ]:
def multiply_back0(grad_out, out, x, y) -> Tensor:
    # d(x*y)/dx = y, chain rule => grad_out * y; then collapse any
    # broadcast axes back to x's original shape.
    if not isinstance(y, Tensor):
        y = t.tensor(y)
    return unbroadcast(grad_out * y, x)


def multiply_back1(grad_out, out, x, y) -> Tensor:
    # d(x*y)/dy = x, chain rule => grad_out * x.
    if not isinstance(x, Tensor):
        x = t.tensor(x)
    return unbroadcast(grad_out * x, y)


<details><summary>Solution</summary>

```python
def multiply_back0(grad_out, out, x, y) -> Tensor:
    # d(x*y)/dx = y, chain rule => grad_out * y; then collapse any
    # broadcast axes back to x's original shape.
    if not isinstance(y, Tensor):
        y = t.tensor(y)
    return unbroadcast(grad_out * y, x)


def multiply_back1(grad_out, out, x, y) -> Tensor:
    # d(x*y)/dy = x, chain rule => grad_out * x.
    if not isinstance(x, Tensor):
        x = t.tensor(x)
    return unbroadcast(grad_out * x, y)
```

**Why unbroadcast matters even on this simple op.** PyTorch lets `x * y` broadcast — `(1,4) * (3,4) -> (3,4)`. The reverse pass computes `grad_out * y` which has shape `(3,4)`, but the parent `x` is `(1,4)`. Without `unbroadcast`, you'd try to accumulate a `(3,4)` grad into a `(1,4)` `.grad` slot and crash.

**Coercing the scalar.** `multiply(t, 3.0)` is allowed — the float isn't a `Tensor`. `multiply_back0` still needs `y` to be tensor-like for the multiplication. The `isinstance` guard handles both paths without forcing the caller to pre-coerce.

**Symmetric registration in BACK_FUNCS.** Even though `multiply` is mathematically symmetric (`x*y == y*x`), we register both `(np.multiply, 0) -> multiply_back0` and `(np.multiply, 1) -> multiply_back1`. The dispatcher always looks up by `(func, argnum)` — it doesn't know which ops are symmetric.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()